In [1]:
%load_ext autoreload
%autoreload 2

print("Autoreload extension loaded. All modules will be reloaded automatically before executing code.")

Autoreload extension loaded. All modules will be reloaded automatically before executing code.


In [ ]:
import logging
import torch
from torchinfo import summary
import lightning.pytorch as pl
from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch.loggers import TensorBoardLogger

from workspace.config import Config
from workspace.dataset import MNISTDataset
from workspace.model import MNISTClassifier

torch.set_float32_matmul_precision("medium")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s"
)

logger = TensorBoardLogger(
    save_dir="data/logs",
    name="deepdive",
)

config = Config()
dataset = MNISTDataset(config)
model = MNISTClassifier(
    layer_depth=config.layer_depth,
    kernel_depth=config.kernel_depth,
    learning_rate=config.learning_rate,
    weight_decay=config.weight_decay,
 )

checkpointer = ModelCheckpoint(
    dirpath=config.checkpoint_rootdir,
    filename="mnist-epoch{epoch:02d}",
    monitor="val_loss",
    mode="min",
    save_last=True,
    save_top_k=1,
)

trainer = pl.Trainer(
    accelerator="cuda",
    devices=[0],
    max_epochs=config.epochs,
    callbacks=[checkpointer],
    deterministic=True,
    enable_progress_bar=True,
    logger=logger,
    precision="16-mixed"
    )

print("\nLoaded", config, "\n")
summary(model, input_size=(config.batch_size, 1, 28, 28), verbose=1)

trainer.fit(model, datamodule=dataset)

validation_metrics = trainer.validate(model, datamodule=dataset, ckpt_path="best", verbose=False)
print("Validation metrics:", validation_metrics)


Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.



Loaded Config {
    "seed": 42,
    "layer_depth": 2,
    "kernel_depth": 8,
    "dataset_rootdir": "data/datasets",
    "dataset_shuffle": true,
    "dataset_transform": "ToTensor()",
    "num_workers": 0,
    "epochs": 10,
    "batch_size": 64,
    "learning_rate": 0.0005,
    "weight_decay": 0.0,
    "checkpoint_rootdir": "data/checkpoints",
    "checkpoint_interval": 1,
    "checkpoint_restore": "last",
    "Torch version:": "2.13.0+cu130",
    "CUDA supported:": true
} 

Layer (type:depth-idx)                        Output Shape              Param #
MNISTClassifier                               [64, 10]                  --
├─Sequential: 1-1                             [64, 32, 7, 7]            --
│    └─MultiScaleBlock: 2-1                   [64, 16, 14, 14]          --
│    │    └─ModuleList: 3-1                   --                        32
│    │    └─Sequential: 3-2                   [64, 16, 28, 28]          176
│    │    └─MaxPool2d: 3-3                    [64, 16, 14, 14]

/opt/venv/lib/python3.12/site-packages/lightning/pytorch/callbacks/model_checkpoint.py:881: Checkpoint directory /workspaces/deepdive/data/checkpoints exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/opt/venv/lib/python3.12/site-packages/lightning/pytorch/utilities/model_summary/model_summary.py:242: Precision 16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.


┏━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name       ┃ Type             ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ criterion  │ CrossEntropyLoss │      0 │ train │     0 │
│ 1 │ features   │ Sequential       │  1.6 K │ train │     0 │
│ 2 │ classifier │ Sequential       │  202 K │ train │     0 │
└───┴────────────┴──────────────────┴────────┴───────┴───────┘

Trainable params: 203 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 203 K                                                                                                
Total estimated model params size (MB): 0.815                                                                      
Modules in train mode: 46                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/opt/venv/lib/python3.12/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` 
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

/opt/venv/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:485: Your 
`val_dataloader`'s sampler has shuffling enabled, it is strongly recommended that you turn shuffling off for 
val/test dataloaders.

/opt/venv/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 
'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the 
`num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.

/opt/venv/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 
'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the 
`num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.